# 极简 Event Tensor Compiler 实现

本文是对于 `Event Tensor: A Unified Abstraction for Compiling Dynamic Megakernel` 论文的一个极简实现，目的是为了理解论文的核心创新点。我将沿着 `Event Tensor` 论文里一个`row-sum`的例子进行讲解，主要分三部分：

1. Event Tensor 基本抽象、最小 runtime primitive 设计，以及本文如何表示依赖关系并进行代码生成
2. Static Scheduling 的逻辑，以及如何用多面体编译技术来实现变化，static task scheduler的实现细节
3. Dynamic Scheduling 的逻辑，以及如何实现 dynamic task scheduler

本文只追求核心逻辑的实现和演示，在lower、codegen过程中一些零碎的代码都放在 `./etensor` 目录，读者们有兴趣可以自行查看。


In [13]:
from pathlib import Path
from dataclasses import dataclass
import tempfile

from IPython.display import Markdown, display
import isl
import numpy as np
import torch
import tilelang.language as T
import tvm_ffi

from etensor import codegen as base_codegen
from etensor import compiler as etc


def code_block(source: str, lang: str = "cpp", limit: int | None = None) -> None:
    text = source if limit is None else source[:limit]
    display(Markdown(f"```{lang}\n{text}\n```"))


## 1. 理解 Event Tensor 与 Task Graph


论文中展示 row-sum 例子，是对于一个列数量为动态的Tensor在行上进行两阶段的reduce，传统的方法是需要分别launch两个kernel，而event tensor compiler的哲学是把这两个kernel的依赖关系表达出来，然后让编译器去调度和生成代码，实现在一个kernel通过遵守依赖的方式来完成计算。

![](images/event_tensor/fig3_event_sample.png)

这里的 row-sum 可以拆成两步, 第一阶段对 `A[n*32, 128]` 在行上按32为tile进行reduce，得到一个 `B[n*32, 4]` 的 Tensor：

$$
B[i, j] = \sum_{k \in [j \cdot 32, j \cdot 32 + 32)} A[i, k]
$$

第二个阶段对 `B[n*32, 4]` 在行上进行reduce，得到一个 `C[n*32]` 的 Tensor：

$$
C[i] = \sum_{j \in [0, 4)} B[i, j]
$$

执行实例 `final_sum(i)` 只依赖 `partial_sum(i, 0) ... partial_sum(i, 3)`，并不依赖别的行，用依赖的方式来表示为：

$$
P[i, j] \rightarrow F[i] \\
\text{P is partial\_sum, F is final\_sum} 
$$


此时把这个依赖关系用一个整型tensor来表示，也就是 Event Tensor：

$$
E[i] = 4 \ \ \\
\text{E is event tensor} 
$$

它的初始值为 4，因为同一个 `F[i]` 要等 4 个partial task 都完成。他的大小会随着 `i` 动态变化。


### 1.1 Event Tensor 实现原理

Event Tensor 的一个元素，本质上就是一个 counter-based synchronization object，通过counter的值来表达同步状态。因此Event Tensor 的 primitive 就是对这个 counter 的操作：

- `notify()`：对 counter 做 atomic decrement。
- `notify_and_ready()`：atomic decrement，并返回这个 event 是否刚好 ready，也就是旧值是否为 1。
- `wait()`：反复读取 counter，直到它变成 0。

我为了保持前端写数组访问的形式，将 API 接口参数设计成 `int* counter` 的形式。所以调用primitive的代码会是这样：
```cpp
etensor::notify(&E0[i]);
etensor::wait(&E0[i]);
```

具体的实现如下:


In [14]:
code_block(Path("./etensor/include/etensor.cuh").read_text(), "cpp")


```cpp
#pragma once

#include <cuda/atomic>
#include <cuda_runtime.h>

namespace etensor {

__device__ __forceinline__ int load_acquire(const int* ptr) {
  cuda::atomic_ref<int, cuda::thread_scope_device> ref(*const_cast<int*>(ptr));
  return ref.load(cuda::memory_order_acquire);
}

__device__ __forceinline__ void store_release(int* ptr, int value) {
  cuda::atomic_ref<int, cuda::thread_scope_device> ref(*ptr);
  ref.store(value, cuda::memory_order_release);
}

__device__ __forceinline__ void notify(int* counter) {
  cuda::atomic_ref<int, cuda::thread_scope_device> ref(*counter);
  ref.fetch_sub(1, cuda::memory_order_release);
}

__device__ __forceinline__ bool notify_and_ready(int* counter) {
  cuda::atomic_ref<int, cuda::thread_scope_device> ref(*counter);
  int old = ref.fetch_sub(1, cuda::memory_order_release);
  return old == 1;
}

__device__ __forceinline__ void wait(int* counter) {
  while (load_acquire(counter) != 0) {
    __nanosleep(64);
  }
}

}  // namespace etensor

```

### 1.2 实现 device function

在论文的例子中，需要有两个 device function 作为 `partial_sum P` 和 `final_sum F`这两个task，每个 task 都会调用 event tensor 的 primitive 来实现同步。 我这里取巧的使用 TileLang 来实现这两个 Task，因为可以借助 TileLang 生成 CUDA source 的能力，把每个 task 编译成 `__device__` 函数，然后塞进 megakernel。


In [15]:
M_TILE = 32
K_TILE = 32
K_SPLIT = 4
K = K_TILE * K_SPLIT


def partial_sum(n: int):
    @T.prim_func
    def partial_task(
        A: T.Tensor((n * M_TILE, K), T.float32),
        B: T.Tensor((n * M_TILE, K_SPLIT), T.float32),
        i: T.int32,
        j: T.int32,
    ):
        with T.Kernel(1, threads=128):
            for r in T.Parallel(M_TILE):
                row = i * M_TILE + r
                acc = T.alloc_local((1,), T.float32)
                acc[0] = 0.0
                for k in T.serial(K_TILE):
                    acc[0] += A[row, j * K_TILE + k]
                B[row, j] = acc[0]

    return partial_task


def final_sum(n: int):
    @T.prim_func
    def final_task(
        B: T.Tensor((n * M_TILE, K_SPLIT), T.float32),
        C: T.Tensor((n * M_TILE,), T.float32),
        i: T.int32,
    ):
        with T.Kernel(1, threads=128):
            for r in T.Parallel(M_TILE):
                row = i * M_TILE + r
                acc = T.alloc_local((1,), T.float32)
                acc[0] = 0.0
                for j in T.serial(K_SPLIT):
                    acc[0] += B[row, j]
                C[row] = acc[0]

    return final_task


### 1.3 自定义 TaskGraph IR

论文中编写了一套 `graph function` 的前端来描述 task 和 event 以及它们之间的关系。我们选择最省力的方式，自定义一个很小的 `TaskGraph` IR，只保留教程需要的几个概念：

- `TaskGraph` 包含 buffer、statement、dependence、schedule 和 placement
  - `Tensor` 描述 tensor 的 shape 和 dtype
  - `Statement` 描述一个 task 的执行点和它的计算逻辑
  - `Dependence` 描述 task 之间的依赖关系
  - `Schedule` 描述 task 的执行顺序
  - `Placement` 描述 task 执行的设备
- `Context` 在jit的时提供静态信息，例如这里把 `n` 固定成 8


In [16]:
@dataclass(frozen=True)
class Tensor:
    domain: isl.set
    dtype: str


@dataclass(frozen=True)
class Statement:
    domain: isl.set
    primfunc: object
    buffers: tuple[str, ...]
    indices: tuple[str, ...]


@dataclass(frozen=True)
class TaskGraph:
    buffers: dict[str, Tensor]
    statements: dict[str, Statement]
    dependence: isl.union_map
    schedule: isl.union_map
    placement: isl.union_map
    context: isl.set | None = None

    def with_context(self, value):
        return value if self.context is None else value.intersect_params(self.context)

    def statement_domain(self, name: str) -> isl.set:
        return self.with_context(self.statements[name].domain)

    def dependence_map(self) -> isl.union_map:
        return self.with_context(self.dependence)

    def schedule_map(self) -> isl.union_map:
        return self.with_context(self.schedule)

    def placement_map(self) -> isl.union_map:
        return self.with_context(self.placement)


接下来我们把论文里 row-sum 例子对应的 TaskGraph IR 构造出来，并且保留它dynamic shape的能力：


In [17]:
def get_task_graph(n: int) -> TaskGraph:
    buffers = {
        "A": Tensor(
            domain=isl.set(f"[n] -> {{ A[m, k] : 0 <= m < {M_TILE}n and 0 <= k < {K} }}"),
            dtype="float",
        ),
        "B": Tensor(
            domain=isl.set(f"[n] -> {{ B[m, j] : 0 <= m < {M_TILE}n and 0 <= j < 4 }}"),
            dtype="float",
        ),
        "C": Tensor(
            domain=isl.set(f"[n] -> {{ C[m] : 0 <= m < {M_TILE}n }}"),
            dtype="float",
        ),
    }
    statements = {
        "P": Statement(
            domain=isl.set(f"[n] -> {{ P[i, j] : 0 <= i < n and 0 <= j < {K_SPLIT} }}"),
            primfunc=partial_sum(n),
            buffers=("A", "B"),
            indices=("i", "j"),
        ),
        "F": Statement(
            domain=isl.set(f"[n] -> {{ F[i] : 0 <= i < n }}"),
            primfunc=final_sum(n),
            buffers=("B", "C"),
            indices=("i",),
        ),
    }
    dependence = isl.union_map(
        f"[n] -> {{ P[i, j] -> F[i] : 0 <= i < n and 0 <= j < {K_SPLIT} }}"
    )
    schedule = isl.union_map(
        f"[n] -> {{ P[i, j] -> [i, 0, j] : 0 <= i < n and 0 <= j < {K_SPLIT}; F[i] -> [i, 1, 0] : 0 <= i < n }}"
    )
    placement = isl.union_map(
        f"[n] -> {{ [i, t, j] -> BlockIdx[x, y] : x = i and 0 <= i < n and j = y and 0 <= j < {K_SPLIT} }}"
    )
    return TaskGraph(
        buffers=buffers,
        statements=statements,
        dependence=dependence,
        schedule=schedule,
        placement=placement,
        context=isl.set(f"[n] -> {{ : n = {n} }}"),
    )


实例化一个n=8的 TaskGraph：



In [18]:
graph = get_task_graph(8)
graph


TaskGraph(buffers={'A': Tensor(domain=isl.set("[n] -> { A[m, k] : 0 <= m < 32n and 0 <= k <= 127 }"), dtype='float'), 'B': Tensor(domain=isl.set("[n] -> { B[m, j] : 0 <= m < 32n and 0 <= j <= 3 }"), dtype='float'), 'C': Tensor(domain=isl.set("[n] -> { C[m] : 0 <= m < 32n }"), dtype='float')}, statements={'P': Statement(domain=isl.set("[n] -> { P[i, j] : 0 <= i < n and 0 <= j <= 3 }"), primfunc=tir.PrimFunc(span=None, struct_info_=relax.FuncStructInfo(span=None, params=(relax.TensorStructInfo(span=None, shape=relax.expr.ShapeExpr(span=None, struct_info_=relax.ShapeStructInfo(span=None, values=(ir.IntImm(span=None, dtype=int64, value=256), ir.IntImm(span=None, dtype=int64, value=128)), ndim=2), values=(ir.IntImm(span=None, dtype=int64, value=256), ir.IntImm(span=None, dtype=int64, value=128))), dtype=float32, vdevice=None, ndim=2), relax.TensorStructInfo(span=None, shape=relax.expr.ShapeExpr(span=None, struct_info_=relax.ShapeStructInfo(span=None, values=(ir.IntImm(span=None, dtype=int64

拿到 task graph 后，我们需要把task间的依赖关系转换为 event tensor 的抽象，我在这里引入 `EventInfo` 的数据结构，它是从 TaskGraph 里的 dependence 衍生出来的一个数据结构，专门用来描述 event tensor 相关信息。

这里dependence是他的对应的依赖关系，这里的domain代表了event tensor的shape，init_values这里表示初始值，也就是consumer需要等待多少个producer完成。这里我特意做成了一个ndarray，是考虑到复杂的例子下，不同的consumer可能需要等待不同数量的producer完成。 另外这里producer和consumer只是单纯的匹配task name，最后的notify_access和wait_access实际是为了描述Producer和Consumer应该访问event tensor的哪个元素，后续codegen会用到。


In [19]:
@dataclass(frozen=True)
class EventInfo:
    name: str
    dependence: isl.map
    domain: isl.set
    init_values: np.ndarray
    producer: str
    consumer: str
    notify_access: isl.map
    wait_access: isl.map


下面需要从TaskGraph IR 里把这些信息提取出来，构造出 `EventInfo`，其实很简单，每个dependence的 range就是 event tensor的shape，然后在每个dependence上统计一下每个consumer需要等待多少个producer，就得到了event tensor的初始值。最后把producer和consumer的task name记录一下，构建出 `EventInfo`。


In [20]:
def derive_event_infos(graph: TaskGraph) -> tuple[EventInfo, ...]:
    events = []
    maps = graph.dependence_map().get_map_list()
    for i in range(maps.n_map()):
        dep = maps.get_at(i)
        name = f"E{i}"
        domain = dep.range()
        lows = [domain.dim_min_val(dim).get_num_si() for dim in range(domain.dim(isl.dim_type.SET))]
        shape = tuple(
            domain.dim_max_val(dim).get_num_si() - lows[dim] + 1
            for dim in range(domain.dim(isl.dim_type.SET))
        )
        init_values = np.zeros(shape, dtype=np.int32)

        wait_access = domain.identity().set_tuple_name(isl.dim_type.OUT, name)
        pma = wait_access.as_pw_multi_aff()
        for dim, low in enumerate(lows):
            if low != 0:
                pma = pma.set_at(dim, pma.at(dim).add_constant(-low))
        wait_access = pma.as_map()

        def fill(point: isl.point) -> None:
            index = tuple(
                point.get_coordinate_val(isl.dim_type.SET, dim).get_num_si() - lows[dim]
                for dim in range(point.dim(isl.dim_type.SET))
            )
            init_values[index] = dep.intersect_range(point.to_set()).domain().count_val().get_num_si()

        domain.foreach_point(fill)
        events.append(
            EventInfo(
                name=name,
                dependence=dep,
                domain=domain,
                init_values=init_values,
                producer=dep.get_tuple_name(isl.dim_type.IN),
                consumer=dep.get_tuple_name(isl.dim_type.OUT),
                notify_access=dep.apply_range(wait_access),
                wait_access=wait_access,
            )
        )
    return tuple(events)


events = derive_event_infos(graph)
events


(EventInfo(name='E0', dependence=isl.map("[n] -> { P[i, j] -> F[i] : n = 8 and 0 <= i <= 7 and 0 <= j <= 3 }"), domain=isl.set("[n] -> { F[i0] : n = 8 and 0 <= i0 <= 7 }"), init_values=array([4, 4, 4, 4, 4, 4, 4, 4], dtype=int32), producer='P', consumer='F', notify_access=isl.map("[n] -> { P[i, j] -> E0[i] : n = 8 and 0 <= i <= 7 and 0 <= j <= 3 }"), wait_access=isl.map("[n] -> { F[i0] -> E0[i0] : n = 8 and 0 <= i0 <= 7 }")),)

### 1.4 从TaskGraph到CodeGen

得到TaskGraph后，我打算使用最naive的方式将它所表示的逻辑进行代码生成，那么就是复用isl的 AST builder，通过marker的方式标记每个 statement instance， 当 statement 打印时，同时打印对应的 `notify` 或 `wait` 。

这里最重要的是 isl AST builder 的两个 hook：

- `at_each_domain`：isl 每生成一个 statement instance，就调用它。我们在这里知道当前打印的是 `P` 还是 `F`。
- `print_user`：真正把这个 statement 打印成 CUDA 代码。我们在这里把 `wait/task/notify` 按顺序输出。

其中 marker 本身仍然只表示“这里要插同步”。在之前的 `EventInfo` 的计算过程中，我们已经把 `P[i, j] -> F[i]` 这个依赖关系转化成了两条 access map，因此具体访问哪个 `E[...]`，由当前 statement instance apply 对应 access map 得到。

```text
notify_access: P[i, j] -> E0[i]
wait_access:   F[i]    -> E0[i]
```

下面的 `render_block_schedule` 就是这个过程的核心：先用 isl 生成 block 内的 statement AST，再在 `at_each_domain` 里为当前 statement 算出 wait/notify 的 event access，最后通过 `print_user` 把普通 task call 替换成 `wait -> task -> notify` 的 CUDA 代码。


In [21]:
def render_block_schedule(graph: TaskGraph, events: tuple[EventInfo, ...]) -> str:
    schedule = base_codegen.build_block_schedule_tree(graph, events)
    user_infos: dict[int, base_codegen.UserPrintInfo] = {}

    def after_mark_callback(node: isl.ast_node_mark, build: isl.ast_build) -> isl.ast_node:
        child = node.node()
        return isl.ast_node_block(isl.ast_node_list(isl.ast_node(child)))

    def at_each_domain(node: isl.ast_node_user, build: isl.ast_build) -> isl.ast_node:
        expr = node.expr()
        statement = expr.get_arg(0).get_id().get_name()
        waits = tuple(
            base_codegen.EventCall(
                event.name,
                base_codegen._event_args_for_current_instance(event.wait_access, build),
            )
            for event in events
            if statement == event.consumer
        )
        notifies = tuple(
            base_codegen.EventCall(
                event.name,
                base_codegen._event_args_for_current_instance(event.notify_access, build),
            )
            for event in events
            if statement == event.producer
        )

        annotation = isl.id(f"{statement}_{len(user_infos)}")
        user_infos[annotation.ptr] = base_codegen.UserPrintInfo(statement, waits, notifies)
        return node.set_annotation(annotation)

    def print_user(printer, options, node):
        expr = node.expr()
        info = user_infos[node.annotation().ptr]

        for event_call in info.waits:
            printer = base_codegen._print_event_call(printer, event_call, "wait")
        if info.waits:
            printer.start_line(); printer.print_str("__syncthreads();"); printer.end_line()

        printer.start_line()
        printer.print_str(f"{info.statement}(")
        for i in range(1, expr.get_n_arg()):
            if i != 1:
                printer.print_str(", ")
            printer.print_ast_expr(expr.get_arg(i))
        printer.print_str(");")
        printer.end_line()

        if info.notifies:
            printer.start_line(); printer.print_str("__syncthreads();"); printer.end_line()
            for event_call in info.notifies:
                printer = base_codegen._print_event_call(printer, event_call, "notify")
        return printer

    fd, raw_path = tempfile.mkstemp(suffix=".c")
    path = Path(raw_path)
    try:
        builder = isl.ast_build.from_context(graph.context) if graph.context is not None else isl.ast_build()
        builder = builder.set_after_each_mark(after_mark_callback)
        builder = builder.set_at_each_domain(at_each_domain)
        ast = builder.node_from(schedule)
        printer = isl.printer.to_file_path(str(path)).set_output_format(isl.format.C)
        options = isl.ast_print_options.alloc().set_print_user(print_user)
        ast.print(printer, options).flush()
        return path.read_text().strip()
    finally:
        path.unlink(missing_ok=True)


schedule_code = render_block_schedule(graph, events)


In [22]:
code_block(schedule_code, "cpp")


```cpp
if (bx >= 0 && bx <= 7 && by >= 0 && by <= 3) {
  P(bx, by);
  __syncthreads();
  if (threadIdx.x == 0) { etensor::notify(&E0[bx]); }
  if (by == 0) {
    if (threadIdx.x == 0) { etensor::wait(&E0[bx]); }
    __syncthreads();
    F(bx);
  }
}
```

当核心代码已经构建，剩下就是添加 host 代码、插入头文件、渲染模板以及编译动态库等细节。这些都放在 `./etensor` 目录里了，这里直接调用 `render_base_cuda_source` 来生成最终的代码：


In [23]:
base_cuda = etc.render_base_cuda_source(graph, events, schedule_code=schedule_code)
code_block(base_cuda, "cpp", limit=4000)


2026-06-05 21:52:09  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:52:09  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.


```cpp
// Auto Generated

#include <cuda_runtime.h>
#include <tvm/ffi/tvm_ffi.h>

#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#include "include/etensor.cuh"

namespace {

__device__ __forceinline__ void etensor_P_task(const float* __restrict__ A, float* __restrict__ B, int i, int j) {
  float acc[1];
  if (((int)threadIdx.x) < 32) {
    acc[0] = 0x0p+0f/*0.000000e+00*/;
    for (int k = 0; k < 32; ++k) {
      float condval;
      if (((((0 <= j) && (j < 4)) && (0 <= i)) && (i < 8))) {
        condval = A[((((((int64_t)i) * (int64_t)4096) + (((int64_t)((int)threadIdx.x)) * (int64_t)128)) + (((int64_t)j) * (int64_t)32)) + ((int64_t)k))];
      } else {
        condval = 0x0p+0f/*0.000000e+00*/;
      }
      acc[0] = (acc[0] + condval);
    }
    if (0 <= j) {
      if (j < 4) {
        if (0 <= i) {
          if (i < 8) {
            B[(((i * 128) + (((int)threadIdx.x) * 4)) + j)] = acc[0];
          }
        }
      }
    }
  }
}

__device__ __forceinline__ void etensor_F_task(const float* __restrict__ B, float* __restrict__ C, int i) {
  float acc[1];
  if (((int)threadIdx.x) < 32) {
    acc[0] = 0x0p+0f/*0.000000e+00*/;
    for (int j = 0; j < 4; ++j) {
      float condval;
      if (((0 <= i) && (i < 8))) {
        condval = B[(((((int64_t)i) * (int64_t)128) + (((int64_t)((int)threadIdx.x)) * (int64_t)4)) + ((int64_t)j))];
      } else {
        condval = 0x0p+0f/*0.000000e+00*/;
      }
      acc[0] = (acc[0] + condval);
    }
    if (0 <= i) {
      if (i < 8) {
        C[((i * 32) + ((int)threadIdx.x))] = acc[0];
      }
    }
  }
}

#define P(i, j) do { etensor_P_task(A, B, i, j); } while (0)
#define F(i) do { etensor_F_task(B, C, i); } while (0)

#define CUDA_CHECK(expr)                                                       \
  do {                                                                         \
    cudaError_t _err = (expr);                                                 \
    if (_err != cudaSuccess) {                                                 \
      std::fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__,      \
                   cudaGetErrorString(_err));                                  \
      std::abort();                                                            \
    }                                                                          \
  } while (0)

__global__ void mega_kernel(float* A, float* B, float* C, int* E0) {
  const int bx = blockIdx.x;
  const int by = blockIdx.y;
  if (bx >= 0 && bx <= 7 && by >= 0 && by <= 3) {
    P(bx, by);
    __syncthreads();
    if (threadIdx.x == 0) { etensor::notify(&E0[bx]); }
    if (by == 0) {
      if (threadIdx.x == 0) { etensor::wait(&E0[bx]); }
      __syncthreads();
      F(bx);
    }
  }
}

#undef P
#undef F

void run_static_impl(int64_t A_addr, int64_t B_addr, int64_t C_addr) {
  float* A = reinterpret_cast<float*>(A_addr);
  float* B = reinterpret_cast<float*>(B_addr);
  float* C = reinterpret_cast<float*>(C_addr);
  int* E0_storage = nullptr;
  std::vector<int> E0_init = { 4, 4, 4, 4, 4, 4, 4, 4 };
  CUDA_CHECK(cudaMalloc(&E0_storage, sizeof(int) * 8));
  CUDA_CHECK(cudaMemcpy(E0_storage, E0_init.data(), sizeof(int) * 8, cudaMemcpyHostToDevice));
  auto E0 = reinterpret_cast<int*>(E0_storage);
  mega_kernel<<<dim3(8, 4), 128>>>(A, B, C, E0);
  CUDA_CHECK(cudaGetLastError());
  CUDA_CHECK(cudaDeviceSynchronize());
  CUDA_CHECK(cudaFree(E0_storage));
}

}  // namespace

TVM_FFI_DLL_EXPORT_TYPED_FUNC(run_static, run_static_impl);
```

### 1.5 Build and Run

最后把基础版 megakernel 编译成动态库，并用同一组输入和 PyTorch reference 对比结果。这里的 `run_row_sum_lib` 后面 static scheduling 和 dynamic scheduling 也会复用。


In [24]:
def run_row_sum_lib(lib_path: Path, n: int, label: str) -> None:
    mod = tvm_ffi.load_module(str(lib_path))
    rows = n * M_TILE
    row = torch.arange(rows, device="cuda", dtype=torch.float32).reshape(rows, 1)
    col = torch.arange(K, device="cuda", dtype=torch.float32).reshape(1, K)
    A = torch.remainder(row, 17) * 0.1 + torch.remainder(col, 11) * 0.01
    B = torch.empty((rows, K_SPLIT), device="cuda", dtype=torch.float32)
    C = torch.empty((rows,), device="cuda", dtype=torch.float32)
    ref = A.sum(dim=1)

    func = mod.get_function("run_static")
    func(A.data_ptr(), B.data_ptr(), C.data_ptr())

    max_abs_diff = torch.max(torch.abs(C - ref)).item()
    print(f"{label} max_abs_diff={max_abs_diff:.6f}")
    if max_abs_diff > 1e-4:
        raise RuntimeError(f"{label} verification failed")
    print("PASS")


base_prefix = Path("etensor/build/tutorial_base_row_sum").resolve()
base_lib = base_codegen.build(graph, base_prefix)
run_row_sum_lib(base_lib, n=8, label="base")


2026-06-05 21:52:09  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:52:09  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
make: Entering directory '<repo>/etensor'
nvcc -O3 -std=c++17 -shared -Xcompiler -fPIC -gencode arch=compute_80,code=compute_80 -I. -I<python-site-packages>/tvm_ffi/include build/tutorial_base_row_sum.cu -o build/tutorial_base_row_sum.so -L<python-site-packages>/tvm_ffi/lib/ -ltvm_ffi -Xlinker -rpath -Xlinker <python-site-packages>/tvm_ffi/lib/
make: Leaving directory '<repo>/etensor'
base max_abs_diff=0.000031
PASS


## 2. Event Tensor编译优化: Static scheduling

论文除了提出了上述抽象，更重要的是提供了一套编译优化的方案。 首先是Static scheduling静态调度， 通过在 launch 前显式把 task 分配到不同 SM 的 task queue 中，task 依赖通过event tensor的 primitive 管理，将多个 device function 融合为同一个函数:

![](images/event_tensor/fig6_static_schedule_before_after.png)

上文说法或许有点抽象，具体而言就是把由于计算资源需求不同的device function通过软件调度的方法转换为**一个计算资源固定的megakernel形式*，这对于许多DSA架构来说相当重要。 比如原本每一个task他单独launch的时候，可以设定大于硬件资源的并行数量，通过gpu block schduler来调度他们的执行，而转换到megakernel时，硬件没办法同时正确调度多个task的执行同时保证他们的依赖关系，因此需要将megakernel以persistent的形式运行，通过软件预先调度task的执行顺序。

在多面体编译的术语下，这叫做 [**space-time mapping**](https://www.infosun.fim.uni-passau.de/publications/docs/GFL04ccpe.pdf)，把 computation point 到 processor/time 的分配，此时是把 task instance 映射到 (CTA/SM worker, step) 上。

优化前，schedule 的 target 还是逻辑执行空间：

```text
task instance -> logical time / tile coordinate
blockIdx      -> logical tile coordinate
```

优化后，schedule 被 lowering 到 processor-time space：

```text
task instance -> (worker, local_step)
blockIdx      -> worker id
local_step    -> worker 本地 queue 中的顺序
```

因此，`blockIdx` 不再表示原始 computation domain 里的 tile 坐标，而是表示一个物理执行资源。原来的逻辑 tile 坐标被 materialize 到 task descriptor 中，由 scheduler 在循环里取出。


### 2.1 modulo based static scheduling 实现

下面用 isl 来实现一个最简单的 modulo based 调度变换。这里的变换可以直接写成三段关系的 composition：

```text
task instance -> logical time -> linear time -> R[worker, step]
```

其中 `logical time` 来自原始 `TaskGraph.schedule`，`linear time` 是为了得到一个全局顺序，`R[worker, step]` 则是 processor-time space。最后反过来用 `R[worker, step] -> task instance` materialize 每个 worker 的 queue。

需要注意的是，`time_to_linear` 只应该作用在真实存在的 logical time 上。比如 row-sum 里 `F[i]` 只有 `[i, 1, 0]`，不存在 `[i, 1, 1..3]`。因此这里会用 `task_to_time.range()` 限制 `time_to_linear` 的 domain。


In [25]:
WORKERS = 4


@dataclass(frozen=True)
class TaskInstance:
    statement: str
    indices: tuple[int, ...]


def point_tuple(point: isl.point) -> tuple[int, ...]:
    return tuple(
        point.get_coordinate_val(isl.dim_type.SET, dim).get_num_si()
        for dim in range(point.dim(isl.dim_type.SET))
    )


def static_resource_map(graph: TaskGraph, worker_count: int) -> isl.union_map:
    task_to_time = graph.schedule_map()
    valid_time = task_to_time.range().as_set()

    phase_stride = valid_time.dim_max_val(2).get_num_si() + 1
    row_stride = phase_stride + 1
    time_to_linear = isl.map(
        f"{{ [i, phase, j] -> T[t] : "
        f"t = i * {row_stride} + phase * {phase_stride} + j "
        f"}}"
    ).intersect_domain(valid_time)

    linear_to_resource = isl.map(
        f"{{ T[t] -> R[worker, step] : "
        f"t = step * {worker_count} + worker and 0 <= worker < {worker_count} "
        f"}}"
    )

    return (task_to_time.
                apply_range(time_to_linear).
                apply_range(linear_to_resource))


def single_task(task_set: isl.union_set) -> TaskInstance:
    tasks = []

    def visit_set(s: isl.set) -> None:
        statement = s.get_tuple_name()
        s.foreach_point(lambda point: tasks.append(TaskInstance(statement, point_tuple(point))))

    task_set.foreach_set(visit_set)
    if len(tasks) != 1:
        raise ValueError(f"expected one task instance, got {task_set}")
    return tasks[0]


def materialize_static_queues(
    task_to_resource: isl.union_map,
    worker_count: int,
) -> tuple[tuple[TaskInstance, ...], ...]:
    resource_to_task = task_to_resource.reverse()
    resource_domain = resource_to_task.domain().as_set()
    queues = []

    for worker in range(worker_count):
        worker_domain = resource_domain.intersect(isl.set(f"{{ R[{worker}, step] }}"))
        steps = []
        worker_domain.foreach_point(
            lambda point: steps.append(
                point.get_coordinate_val(isl.dim_type.SET, 1).get_num_si()
            )
        )

        queue = []
        for step in sorted(steps):
            resource_point = isl.union_set(f"{{ R[{worker}, {step}] }}")
            task_set = resource_to_task.intersect_domain(resource_point).range()
            queue.append(single_task(task_set))
        queues.append(tuple(queue))

    return tuple(queues)


task_to_resource = static_resource_map(graph, WORKERS)
static_queues = materialize_static_queues(task_to_resource, WORKERS)

print("task_to_resource:")
print(task_to_resource)
print()
for worker, queue in enumerate(static_queues):
    print("worker", worker, [(x.statement, x.indices) for x in queue[:8]])


task_to_resource:
[n] -> { P[i, j] -> R[worker, step] : n = 8 and 4step = 5i + j - worker and 0 <= i <= 7 and 0 <= j <= 3 and 0 <= worker <= 3; F[i] -> R[worker, step] : n = 8 and 4step = 4 + 5i - worker and 0 <= i <= 7 and 0 <= worker <= 3 }

worker 0 [('P', (0, 0)), ('F', (0,)), ('P', (1, 3)), ('P', (2, 2)), ('P', (3, 1)), ('P', (4, 0)), ('F', (4,)), ('P', (5, 3))]
worker 1 [('P', (0, 1)), ('P', (1, 0)), ('F', (1,)), ('P', (2, 3)), ('P', (3, 2)), ('P', (4, 1)), ('P', (5, 0)), ('F', (5,))]
worker 2 [('P', (0, 2)), ('P', (1, 1)), ('P', (2, 0)), ('F', (2,)), ('P', (3, 3)), ('P', (4, 2)), ('P', (5, 1)), ('P', (6, 0))]
worker 3 [('P', (0, 3)), ('P', (1, 2)), ('P', (2, 1)), ('P', (3, 0)), ('F', (3,)), ('P', (4, 3)), ('P', (5, 2)), ('P', (6, 1))]


### 2.2 Static Scheduler Runtime 实现

上面已经通过 isl relation composition 得到了每个 worker 的task queue。接下来要把它 lowering 成 CUDA 里能直接消费的数据结构。

static scheduler 需要三个 constant array：

```cpp
static_task_indices[]  // 所有 task 的 index 展平后的数组，例如 P 需要两个 index，F 需要一个 index
static_tasks[]         // 每个 task 的 task_type 和 index_begin
static_queues[]        // 每个 worker 的 task_begin/task_end
```

具体执行时：

- `static_queues[worker id]` 告诉当前 worker 应该消费哪一段 task。
- `static_tasks[task_pos]` 告诉 scheduler 当前 task 是 `P` 还是 `F`，以及它的 index 从哪里开始。
- `static_task_indices + index_begin` 就是 task 的逻辑坐标指针。对 `P` 来说可以读 `task_idx[0], task_idx[1]`；对 `F` 来说只读 `task_idx[0]`。

persistent kernel 里面每个 worker 做的事情很简单，它没有全局抢占逻辑，也没有 ready queue，只是每个 worker 顺序扫描自己那段 static queue。

```cpp
StaticTaskScheduler scheduler;
scheduler.init();
while (scheduler.valid()) {
  const int* task_idx = scheduler.indices();
  switch (scheduler.type()) { ... }
}
```


In [26]:
code_block(Path("./etensor/include/static_tile_scheduler.cuh").read_text(), "cpp")


```cpp
#pragma once

struct StaticTaskScheduler {
  int task_pos;
  int task_end;
  int task_type;
  const int* task_idx;

  __device__ void init() {
    StaticQueueDesc queue = static_queues[blockIdx.x];
    task_pos = queue.task_begin;
    task_end = queue.task_end;
    task_type = -1;
    task_idx = nullptr;
  }

  __device__ bool valid() {
    if (task_pos >= task_end) {
      return false;
    }

    StaticTaskDesc task = static_tasks[task_pos++];
    task_type = task.task_type;
    task_idx = static_task_indices + task.index_begin;
    return true;
  }

  __device__ int type() const {
    return task_type;
  }

  __device__ const int* indices() const {
    return task_idx;
  }
};

```

### 2.3 Static Schedule CodeGen

把上述的调度过程串联起来执行，就是把一个TaskGraph转换为一个StaticScheduledGraph，然后将对应的task instance materialize到每个worker自己的queue中，然后配合一些琐碎的代码生成工作:


In [27]:
@dataclass(frozen=True)
class StaticQueue:
    task_kinds: tuple[int, ...]
    task_indices: tuple[int, ...]


@dataclass(frozen=True)
class StaticScheduledGraph:
    source: TaskGraph
    events: tuple[EventInfo, ...]
    worker_count: int
    task_kinds: dict[str, int]
    task_index_ranks: tuple[int, ...]
    queues: tuple[StaticQueue, ...]


def static_schedule(graph: TaskGraph, worker_count: int) -> StaticScheduledGraph:
    task_kinds = {name: i for i, name in enumerate(graph.statements)}
    task_index_ranks = tuple(len(statement.indices) for statement in graph.statements.values())
    queues = []
    task_to_resource = static_resource_map(graph, worker_count)
    for queue in materialize_static_queues(task_to_resource, worker_count):
        kinds = tuple(task_kinds[x.statement] for x in queue)
        indices = tuple(v for x in queue for v in x.indices)
        queues.append(StaticQueue(kinds, indices))
    return StaticScheduledGraph(graph, events, worker_count, task_kinds, task_index_ranks, tuple(queues))


static_plan = static_schedule(graph, worker_count=WORKERS)
for array in etc.build_static_queue_arrays(static_plan):
    print(array.name, "=", array.values[:160], "...")

static_cuda = etc.render_static_cuda_source(static_plan)
static_core_begin = static_cuda.index("enum TaskKind : int")
static_core_end = static_cuda.index("\n\n#undef", static_cuda.index("__global__ void mega_kernel"))
code_block(static_cuda[static_core_begin:static_core_end], "cpp")


static_task_indices = 0, 0, 0, 1, 3, 2, 2, 3, 1, 4, 0, 4, 5, 3, 6, 2, 7, 1, 0, 1, 1, 0, 1, 2, 3, 3, 2, 4, 1, 5, 0, 5, 6, 3, 7, 2, 0, 2, 1, 1, 2, 0, 2, 3, 3, 4, 2, 5, 1, 6, 0, 6, 7, 3 ...
static_tasks = { 0, 0 }, { 1, 2 }, { 0, 3 }, { 0, 5 }, { 0, 7 }, { 0, 9 }, { 1, 11 }, { 0, 12 }, { 0, 14 }, { 0, 16 }, { 0, 18 }, { 0, 20 }, { 1, 22 }, { 0, 23 }, { 0, 25 }, { ...
static_queues = { 0, 10 }, { 10, 20 }, { 20, 30 }, { 30, 40 } ...
2026-06-05 21:52:35  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:52:35  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.


```cpp
enum TaskKind : int {
TASK_P = 0,
TASK_F = 1,
};

struct StaticTaskDesc {
  int task_type;
  int index_begin;
};

struct StaticQueueDesc {
  int task_begin;
  int task_end;
};

__device__ __constant__ int static_task_indices[] = { 0, 0, 0, 1, 3, 2, 2, 3, 1, 4, 0, 4, 5, 3, 6, 2, 7, 1, 0, 1, 1, 0, 1, 2, 3, 3, 2, 4, 1, 5, 0, 5, 6, 3, 7, 2, 0, 2, 1, 1, 2, 0, 2, 3, 3, 4, 2, 5, 1, 6, 0, 6, 7, 3, 0, 3, 1, 2, 2, 1, 3, 0, 3, 4, 3, 5, 2, 6, 1, 7, 0, 7 };
__device__ __constant__ StaticTaskDesc static_tasks[] = { { 0, 0 }, { 1, 2 }, { 0, 3 }, { 0, 5 }, { 0, 7 }, { 0, 9 }, { 1, 11 }, { 0, 12 }, { 0, 14 }, { 0, 16 }, { 0, 18 }, { 0, 20 }, { 1, 22 }, { 0, 23 }, { 0, 25 }, { 0, 27 }, { 0, 29 }, { 1, 31 }, { 0, 32 }, { 0, 34 }, { 0, 36 }, { 0, 38 }, { 0, 40 }, { 1, 42 }, { 0, 43 }, { 0, 45 }, { 0, 47 }, { 0, 49 }, { 1, 51 }, { 0, 52 }, { 0, 54 }, { 0, 56 }, { 0, 58 }, { 0, 60 }, { 1, 62 }, { 0, 63 }, { 0, 65 }, { 0, 67 }, { 0, 69 }, { 1, 71 } };
__device__ __constant__ StaticQueueDesc static_queues[] = { { 0, 10 }, { 10, 20 }, { 20, 30 }, { 30, 40 } };

#include "include/static_tile_scheduler.cuh"

#define CUDA_CHECK(expr)                                                       \
  do {                                                                         \
    cudaError_t _err = (expr);                                                 \
    if (_err != cudaSuccess) {                                                 \
      std::fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__,      \
                   cudaGetErrorString(_err));                                  \
      std::abort();                                                            \
    }                                                                          \
  } while (0)

__global__ void mega_kernel(float* A, float* B, float* C, int* E0) {
  StaticTaskScheduler scheduler;
  scheduler.init();

  while (scheduler.valid()) {
    const int* task_idx = scheduler.indices();
    switch (scheduler.type()) {
    case TASK_P: {
      const int i = task_idx[0];
      const int j = task_idx[1];
      P(i, j);
      __syncthreads();
      if (threadIdx.x == 0) { etensor::notify(&E0[i]); }
      break;
    }
    case TASK_F: {
      const int i = task_idx[0];
      if (threadIdx.x == 0) { etensor::wait(&E0[i]); }
      __syncthreads();
      F(i);
      break;
    }
    }
  }
}
```

### 2.4 Build and Run

static scheduling 版本应该和基础版得到相同结果。区别只在于 kernel launch 变成固定数量的 persistent worker，task 坐标来自前面生成的 static queue。


In [28]:
from etensor import static_queue_codegen

static_prefix = Path("etensor/build/tutorial_static_row_sum").resolve()
static_lib = static_queue_codegen.build(static_plan, static_prefix)
run_row_sum_lib(static_lib, n=8, label="static_queue")


2026-06-05 21:52:35  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:52:35  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
make: Entering directory '<repo>/etensor'
nvcc -O3 -std=c++17 -shared -Xcompiler -fPIC -gencode arch=compute_80,code=compute_80 -I. -I<python-site-packages>/tvm_ffi/include build/tutorial_static_row_sum.cu -o build/tutorial_static_row_sum.so -L<python-site-packages>/tvm_ffi/lib/ -ltvm_ffi -Xlinker -rpath -Xlinker <python-site-packages>/tvm_ffi/lib/
make: Leaving directory '<repo>/etensor'
static_queue max_abs_diff=0.000031
PASS


## 3. Event Tensor 编译优化: Dynamic scheduling

我理解的 Dynamic scheduling 是一个 **online schedule construction**：编译器不再提前决定每个 logical task 由哪个 worker 在第几个 local step 执行，而是只生成 task set、依赖触发规则和 scheduler 数据结构；实际的 `(worker, local_step)` 映射在运行时由 pop 操作决定:

![](images/event_tensor/fig7_dynamic_schedule_after.png)

以 row-sum 举例，列出它的执行流程：

1. 初始化时只 push `P` task set。
2. `P` set 被领取完后 push `F` task set。
3. `F[i]` 内部仍然 wait `E[i]`，保证每一行的依赖正确。

论文里的完整 dynamic scheduling 是 event-triggered：当某个 event counter 变成 0，就把对应 consumer task push 进 scheduler。



### 3.1 Dynamic schedule transform

Static scheduling 生成的是每个 worker 的固定 queue；dynamic scheduling 生成的是 task set。一个 task set 可以理解成“一批同类型 tile operation 的列表”，runtime 只维护这个 set 当前消费到哪里。实际上执行起来就是，把所有的task instance materialize到一个task set中即可。

比如 row-sum 这个最小例子里有两个 set：

```text
set 0: all P[i, j]
set 1: all F[i]
```

初始化时只有 `P` set ready；当 `P` set 消费结束后，scheduler 把 `F` set 放进 ready queue。


In [29]:
@dataclass(frozen=True)
class DynamicTaskSet:
    task_kind: int
    instances: tuple[TaskInstance, ...]
    next_set_id: int = -1


@dataclass(frozen=True)
class DynamicScheduledGraph:
    source: TaskGraph
    events: tuple[EventInfo, ...]
    worker_count: int
    task_kinds: dict[str, int]
    task_index_ranks: tuple[int, ...]
    task_sets: tuple[DynamicTaskSet, ...]
    initial_ready_sets: tuple[int, ...]


def dynamic_schedule(graph: TaskGraph, worker_count: int) -> DynamicScheduledGraph:
    task_kinds = {name: i for i, name in enumerate(graph.statements)}
    task_index_ranks = tuple(len(statement.indices) for statement in graph.statements.values())
    by_statement = {name: [] for name in graph.statements}
    task_to_resource = static_resource_map(graph, worker_count=1)
    for instance in materialize_static_queues(task_to_resource, worker_count=1)[0]:
        by_statement[instance.statement].append(instance)

    event = events[0]
    task_sets = (
        DynamicTaskSet(task_kinds[event.producer], tuple(by_statement[event.producer]), next_set_id=1),
        DynamicTaskSet(task_kinds[event.consumer], tuple(by_statement[event.consumer])),
    )
    return DynamicScheduledGraph(graph, events, worker_count, task_kinds, task_index_ranks, task_sets, initial_ready_sets=(0,))


dynamic_plan = dynamic_schedule(graph, worker_count=WORKERS)
for set_id, task_set in enumerate(dynamic_plan.task_sets):
    print("set", set_id, "kind", task_set.task_kind, "tiles", [x.indices for x in task_set.instances[:8]], "next", task_set.next_set_id)


set 0 kind 0 tiles [(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3)] next 1
set 1 kind 1 tiles [(0,), (1,), (2,), (3,), (4,), (5,), (6,), (7,)] next -1


### 3.2 Dynamic scheduler runtime

Dynamic scheduler 生成的数据结构可以拆成两层：

```cpp
dynamic_task_indices[]  // 所有 task set 的 logical coordinates，提前展开好
task_sets[]             // 每个 set 的 task_type/index_rank/index_begin/index_end/next_set_id
```

和 static queue 的区别在于：

- static 是 `worker -> task range`，worker 只消费自己的 queue。
- dynamic 是 `ready queue -> task set -> tile cursor`，多个 worker 可以竞争同一个 ready set。

runtime 里还有几组全局状态：

```cpp
dynamic_set_index_pos[set]  // 每个 task set 当前消费到哪个 index
dynamic_ready_queue[]       // 当前 ready 的 task set id 队列
dynamic_queue_head/tail     // ready queue 的 head/tail
dynamic_queue_lock          // 简单 centralized queue lock
dynamic_tiles_done          // 已经 dispatch 出去的 tile 数
```

persistent kernel 的主循环仍然保持和 static scheduler 类似的形状：

```cpp
__shared__ TaskScheduler scheduler;
scheduler.init();
while (scheduler.valid()) {
  const int* task_idx = scheduler.indices();
  switch (scheduler.type()) { ... }
}
```

这里 `scheduler.valid()` 不只是判断条件，它也负责从全局 ready queue 里领取下一个 tile。下面看一下 dynamic runtime scheduler 的实现：


In [30]:
code_block(Path("./etensor/include/dynamic_tile_scheduler.cuh").read_text(), "cpp")

```cpp
#pragma once

#include <cuda_runtime.h>

__device__ int dynamic_set_index_pos[TASK_SET_COUNT];
__device__ int dynamic_ready_queue[READY_QUEUE_CAPACITY];
__device__ int dynamic_queue_head;
__device__ int dynamic_queue_tail;
__device__ int dynamic_queue_lock;
__device__ int dynamic_tiles_done;

__device__ inline void dynamic_lock_queue() {
  while (atomicCAS(&dynamic_queue_lock, 0, 1) != 0) {
    __nanosleep(64);
  }
}

__device__ inline void dynamic_unlock_queue() {
  atomicExch(&dynamic_queue_lock, 0);
}

__device__ inline void push_task_set(int set_id) {
  if (set_id < 0) {
    return;
  }
  dynamic_lock_queue();
  int pos = dynamic_queue_tail++;
  if (pos < READY_QUEUE_CAPACITY) {
    dynamic_ready_queue[pos] = set_id;
  }
  dynamic_unlock_queue();
}

__device__ inline bool pop_task_set(int* set_id) {
  bool found = false;
  dynamic_lock_queue();
  if (dynamic_queue_head < dynamic_queue_tail) {
    *set_id = dynamic_ready_queue[dynamic_queue_head++];
    found = true;
  }
  dynamic_unlock_queue();
  return found;
}

__device__ inline bool dynamic_scheduler_done() {
  return atomicAdd(&dynamic_tiles_done, 0) >= TOTAL_TILE_COUNT;
}

__device__ inline bool pop_tile_from_set(
    int set_id,
    int* task_type,
    const int** task_idx) {
  TaskSetDesc set = task_sets[set_id];
  int index_len = set.index_end - set.index_begin;
  int index_pos = atomicAdd(&dynamic_set_index_pos[set_id], set.index_rank);
  if (index_pos >= index_len) {
    if (index_pos == index_len) {
      push_task_set(set.next_set_id);
    }
    return false;
  }

  *task_type = set.task_type;
  *task_idx = dynamic_task_indices + set.index_begin + index_pos;
  return true;
}

__device__ inline bool pop_tile(int* active_set, int* task_type, const int** task_idx) {
  while (!dynamic_scheduler_done()) {
    if (*active_set >= 0) {
      if (pop_tile_from_set(*active_set, task_type, task_idx)) {
        return true;
      }
      *active_set = -1;
    }

    int set_id = -1;
    if (pop_task_set(&set_id)) {
      *active_set = set_id;
      continue;
    }

    __nanosleep(64);
  }
  return false;
}

struct TaskScheduler {
  int active_set;
  int task_type;
  const int* task_idx;
  int has_task;

  __device__ void init() {
    if (threadIdx.x == 0) {
      active_set = -1;
      task_type = -1;
      task_idx = nullptr;
      has_task = 0;
    }
    __syncthreads();
  }

  __device__ bool valid() {
    if (threadIdx.x == 0) {
      has_task = pop_tile(&active_set, &task_type, &task_idx) ? 1 : 0;
      if (has_task) {
        atomicAdd(&dynamic_tiles_done, 1);
      }
    }
    __syncthreads();
    return has_task != 0;
  }

  __device__ int type() const {
    return task_type;
  }

  __device__ const int* indices() const {
    return task_idx;
  }
};

```

### 3.3 Dynamic CUDA codegen

最后把 task set、dynamic runtime scheduler 和 task dispatch 逻辑放进同一个 persistent kernel。这里同样只截取从 `enum TaskKind` 到 `mega_kernel` 结束的核心部分，避免重复展示前面已经看过的 TileLang device function。


In [31]:
dynamic_cuda = etc.render_dynamic_cuda_source(dynamic_plan)
dynamic_core_begin = dynamic_cuda.index("enum TaskKind")
dynamic_core_end = dynamic_cuda.index("\n\n#undef", dynamic_cuda.index("__global__ void mega_kernel"))
code_block(dynamic_cuda[dynamic_core_begin:dynamic_core_end], "cpp")

2026-06-05 21:53:07  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:53:07  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.


```cpp
enum TaskKind : int {
TASK_P = 0,
TASK_F = 1,
};

constexpr int TASK_SET_COUNT = 2;
constexpr int READY_QUEUE_CAPACITY = 58;
constexpr int TOTAL_TILE_COUNT = 40;

struct TaskSetDesc {
  int task_type;
  int index_rank;
  int index_begin;
  int index_end;
  int next_set_id;
};

__device__ __constant__ int dynamic_task_indices[] = { 0, 0, 0, 1, 0, 2, 0, 3, 1, 0, 1, 1, 1, 2, 1, 3, 2, 0, 2, 1, 2, 2, 2, 3, 3, 0, 3, 1, 3, 2, 3, 3, 4, 0, 4, 1, 4, 2, 4, 3, 5, 0, 5, 1, 5, 2, 5, 3, 6, 0, 6, 1, 6, 2, 6, 3, 7, 0, 7, 1, 7, 2, 7, 3, 0, 1, 2, 3, 4, 5, 6, 7 };
__device__ __constant__ TaskSetDesc task_sets[] = { { 0, 2, 0, 64, 1 }, { 1, 1, 64, 72, -1 } };

#include "include/dynamic_tile_scheduler.cuh"

#define CUDA_CHECK(expr)                                                       \
  do {                                                                         \
    cudaError_t _err = (expr);                                                 \
    if (_err != cudaSuccess) {                                                 \
      std::fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__,      \
                   cudaGetErrorString(_err));                                  \
      std::abort();                                                            \
    }                                                                          \
  } while (0)

__global__ void mega_kernel(float* A, float* B, float* C, int* E0) {
  __shared__ TaskScheduler scheduler;

  scheduler.init();

  while (scheduler.valid()) {
    const int* task_idx = scheduler.indices();
    switch (scheduler.type()) {
    case TASK_P: {
      const int i = task_idx[0];
      const int j = task_idx[1];
      P(i, j);
      __syncthreads();
      if (threadIdx.x == 0) { etensor::notify(&E0[i]); }
      break;
    }
    case TASK_F: {
      const int i = task_idx[0];
      if (threadIdx.x == 0) { etensor::wait(&E0[i]); }
      __syncthreads();
      F(i);
      break;
    }
    }
  }
}
```

### 3.4 Build and Run

dynamic scheduling 版本同样使用 persistent worker，但 task 由 runtime scheduler 从 ready queue 中领取。正确性仍然由 Event Tensor 的 wait/notify 保护。


In [32]:
from etensor import dynamic_queue_codegen

dynamic_prefix = Path("etensor/build/tutorial_dynamic_row_sum").resolve()
dynamic_lib = dynamic_queue_codegen.build(dynamic_plan, dynamic_prefix)
run_row_sum_lib(dynamic_lib, n=8, label="dynamic_queue")


2026-06-05 21:53:07  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'partial_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
2026-06-05 21:53:07  [TileLang:tilelang.cache.kernel_cache:WARNING] (kernel_cache.py:322): Found kernel 'final_task' in memory cache. For better performance, consider using `@tilelang.jit` instead of direct kernel caching.
make: Entering directory '<repo>/etensor'
nvcc -O3 -std=c++17 -shared -Xcompiler -fPIC -gencode arch=compute_80,code=compute_80 -I. -I<python-site-packages>/tvm_ffi/include build/tutorial_dynamic_row_sum.cu -o build/tutorial_dynamic_row_sum.so -L<python-site-packages>/tvm_ffi/lib/ -ltvm_ffi -Xlinker -rpath -Xlinker <python-site-packages>/tvm_ffi/lib/
make: Leaving directory '<repo>/etensor'
dynamic_queue max_abs_diff=0.000031
PASS


## 4. 总结

我觉得这篇论文最有价值的还是两个编译优化pass，把相对独立的task合成到一个megakernel中，是task级别的fusion，不是传统AI编译中的memory fusion，非常有趣。后续如果可以把gmem task和smem task再组合起来优化，效果应该会非常好。 
